# 04 — Mamba: Selective State Space Models

Mamba introduces input-dependent SSM parameters, dramatically improving language modelling performance.

## Selective SSM and the Hardware-Aware Algorithm

**The S4 limitation**: in S4, the *A*, *B*, *C* matrices are input-independent — the same SSM is applied regardless of what the input is. This means the model cannot selectively 'remember' or 'ignore' inputs based on content.

**Mamba's selective SSM** (Gu & Dao, 2023): makes *B*, *C*, and Δ (the step size) functions of the input *u_t*:
$$\Delta_t = \text{softplus}(\tau_\Delta(x_t) + b_\Delta)$$
$$B_t = \tau_B(x_t)$$
$$C_t = \tau_C(x_t)$$
When *Δ* is large for a token, that token gets a large update to the state (the model 'remembers' it). When *Δ* is small, the state barely changes (the model 'ignores' it). This is analogous to the forget/input gates in LSTMs, but derived from first principles.

**Hardware-aware algorithm**: input-dependent SSMs cannot be pre-computed as a fixed convolution (unlike S4). Mamba uses a parallel scan (prefix sum) algorithm that is computed in SRAM (fast) rather than HBM (slow), avoiding the memory bandwidth bottleneck that plagues naive implementations.

**Results**: Mamba matches or exceeds transformer quality on language modelling at the 3B scale while being 5x faster at inference (constant memory, no KV cache).

In [ ]:
# Mamba block from scratch
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

class SelectiveSSM(nn.Module):
    """
    Mamba-style selective SSM.
    B, C, Delta are input-dependent.
    """
    def __init__(self, d_model, d_state=16, expand=2):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_inner = d_model * expand

        # Input projection
        self.in_proj = nn.Linear(d_model, self.d_inner * 2)  # x and z branches

        # SSM parameters
        self.A_log = nn.Parameter(torch.log(torch.arange(1, d_state+1, dtype=torch.float32))
                                   .unsqueeze(0).expand(self.d_inner, -1))
        self.D = nn.Parameter(torch.ones(self.d_inner))

        # Input-dependent projections
        self.x_proj = nn.Linear(self.d_inner, d_state + d_state + 1)  # B, C, delta
        self.dt_proj = nn.Linear(1, self.d_inner)  # project delta to full dim

        # Output
        self.out_proj = nn.Linear(self.d_inner, d_model)

    def ssm(self, x):
        """
        x: (batch, seq_len, d_inner)
        Returns: y of same shape
        """
        B, L, D = x.shape
        A = -torch.exp(self.A_log)  # (D, d_state) — negative for stability

        # Input-dependent parameters
        xBCdt = self.x_proj(x)  # (B, L, d_state*2 + 1)
        B_mat = xBCdt[:, :, :self.d_state]     # (B, L, d_state)
        C_mat = xBCdt[:, :, self.d_state:2*self.d_state]
        delta_log = xBCdt[:, :, -1:]            # (B, L, 1)
        delta = F.softplus(self.dt_proj(delta_log))  # (B, L, D)

        # Discretise A, B
        A_bar = torch.exp(delta.unsqueeze(-1) * A.unsqueeze(0).unsqueeze(0))
        B_bar = delta.unsqueeze(-1) * B_mat.unsqueeze(2)  # (B, L, D, d_state)

        # Selective scan (simple loop — production uses parallel prefix scan)
        h = torch.zeros(B, D, self.d_state)
        ys = []
        for i in range(L):
            h = A_bar[:, i] * h + B_bar[:, i] * x[:, i:i+1, :].unsqueeze(-1).expand_as(B_bar[:, i])
            y_i = (h * C_mat[:, i].unsqueeze(1)).sum(-1)  # (B, D)
            ys.append(y_i)

        y = torch.stack(ys, dim=1)  # (B, L, D)
        return y + self.D * x

    def forward(self, x):
        xz = self.in_proj(x)  # (B, L, 2*D_inner)
        x_branch, z = xz.chunk(2, dim=-1)
        x_branch = F.silu(x_branch)
        y = self.ssm(x_branch)
        y = y * F.silu(z)  # gating
        return self.out_proj(y)

torch.manual_seed(42)
mamba_block = SelectiveSSM(d_model=32, d_state=8)
x = torch.randn(2, 64, 32)
out = mamba_block(x)
print(f'Mamba block output: {out.shape}')
print(f'Params: {sum(p.numel() for p in mamba_block.parameters()):,}')

In [ ]:
# Compare Mamba vs S4 vs Transformer on selective copying
import matplotlib.pyplot as plt

# Selective copy task: copy only the tokens preceded by a marker
def make_selective_copy(batch=32, seq_len=40, vocab=6):
    tokens = torch.randint(1, vocab, (batch, seq_len))
    markers = (torch.rand(batch, seq_len) < 0.3).long()
    inputs = tokens * (1 - markers) + vocab * markers  # vocab = marker token
    targets = tokens * markers  # only marked tokens matter
    return inputs, targets

# Build small Mamba model
class MambaLM(nn.Module):
    def __init__(self, vocab_size=8, d_model=32, d_state=8, n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([SelectiveSSM(d_model, d_state) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        h = self.embed(x)
        for layer in self.layers:
            h = layer(h) + h
        return self.head(self.norm(h))

mamba_lm = MambaLM()
opt = torch.optim.Adam(mamba_lm.parameters(), lr=1e-3)

train_losses = []
for step in range(1000):
    inputs, targets = make_selective_copy()
    logits = mamba_lm(inputs.clamp(0, 7))
    loss = nn.CrossEntropyLoss(ignore_index=0)(logits.view(-1, 8), targets.view(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 200 == 0:
        train_losses.append(loss.item())
        print(f'Step {step}: loss={loss.item():.4f}')

print('\nMamba on selective copy task complete')